# Custom BKT — optimised pipeline

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score, mean_squared_error

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

## 1. Load & pre-process

In [ ]:
# NOTE: Same preprocessing as the analysis notebook, condensed. Reads 25kfiles_kt1.csv.

df = pd.read_csv("25kfiles_kt1.csv")
df = df.drop(columns=["Unnamed: 0"])
df['question_id'] = df['question_id'].str.lstrip('q').astype(int)

In [ ]:
# NOTE: Build the question->skill (KC) map: split tags, then explode to one row per pair.

df_question_content = pd.read_csv("contents\\questions.csv")

df_question_content['tag_list'] = df_question_content['tags'].str.split(';')
df_question_content['question_id_int'] = (
    df_question_content['question_id'].str.lstrip('q').astype(int)
)

questions_exploded = df_question_content.explode('tag_list')
question_skills = (
    questions_exploded[['question_id_int', 'tag_list', 'correct_answer']]
    .drop_duplicates()
    .rename(columns={'question_id_int': 'question_id', 'tag_list': 'skill_name'})
)

In [ ]:
# NOTE: Attach skills + correctness, order each student's attempts, keep the 4 BKT
#       columns. Row count expands per tag, exactly as in the analysis notebook.

df_merged = pd.merge(df, question_skills, on='question_id', how='left')
df_merged['correct'] = (df_merged['user_answer'] == df_merged['correct_answer']).astype(int)

df_merged = df_merged.sort_values(['user_id', 'timestamp'])
df_merged['order_id'] = df_merged.groupby('user_id').cumcount() + 1

df_bkt = df_merged[['user_id', 'skill_name', 'correct', 'order_id']]
df_bkt.head()

## 2. Custom BKT — optimised (Baum-Welch EM)

### What changed and why

| Original | Optimised | Speedup |
|---|---|---|
| Loop over every student sequence one-by-one | **Batch-vectorise** across all sequences of the same length simultaneously | ~29× per skill |
| Sequential skill fitting | **Parallel** skill fitting via `joblib` | ×n_cores |
| Per-group Python loop in `predict_bkt` | Vectorised forward pass over all users per skill | minor |

**How batch vectorisation works:**  
Sequences are grouped by length. Within each length-group the forward pass, backward pass, γ and ξ accumulators are all computed as 3-D NumPy operations `(time, students, states)` — replacing the innermost Python `for seq in seqs` loop with a single set of array ops.  The EM maths is identical; only the execution path changes.

In [ ]:
# ─── forward_pass ─────────────────────────────────────────────────────────────
# (unchanged — still used for prediction one sequence at a time)
def forward_pass(sequence, prior, learn, guess, slip):
    """Filtered P(correct) at each step — used for predictions only."""
    p_l = prior
    predictions = []
    for correct in sequence:
        p_correct = p_l * (1 - slip) + (1 - p_l) * guess
        predictions.append(p_correct)
        if correct == 1:
            num = p_l * (1 - slip)
            den = num + (1 - p_l) * guess
        else:
            num = p_l * slip
            den = num + (1 - p_l) * (1 - guess)
        p_l_given_obs = num / den if den > 0 else p_l
        p_l = p_l_given_obs + (1 - p_l_given_obs) * learn
    return predictions


# ─── fit_skill ────────────────────────────────────────────────────────────────
def fit_skill(sequences, n_iter=100, tol=1e-5,
              init_prior=0.3, init_learn=0.1,
              init_guess=0.2, init_slip=0.1):
    """Fit one skill via Baum-Welch EM — batch-vectorised across sequences.

    Sequences are grouped by length so that all students with the same
    number of attempts are processed together as a (time × students × states)
    array, eliminating the inner Python loop over individual sequences.
    """
    by_length = defaultdict(list)
    for s in sequences:
        if len(s) > 0:
            by_length[len(s)].append(s)
    # Each batch: numpy array of shape (N_students, length)
    batches = [(L, np.array(seqs, dtype=np.int8))
               for L, seqs in by_length.items()]
    if not batches:
        return {'prior': init_prior, 'learn': init_learn,
                'guess': init_guess, 'slip': init_slip,
                'n_sequences': 0, 'n_obs': 0,
                'log_likelihood': float('nan'), 'iterations': 0}

    prior, learn, guess, slip = init_prior, init_learn, init_guess, init_slip
    n_sequences = sum(b.shape[0] for _, b in batches)
    n_obs_total  = sum(L * b.shape[0] for L, b in batches)
    prev_ll = -np.inf

    for it in range(n_iter):
        sum_prior_post  = 0.0
        sum_xi_01       = 0.0
        sum_gamma0_trans = 0.0
        sum_correct_nl  = 0.0
        sum_total_nl    = 0.0
        sum_wrong_l     = 0.0
        sum_total_l     = 0.0
        total_ll        = 0.0
        n_seq           = 0
        A00, A01        = 1 - learn, learn

        for L, batch in batches:          # batch: (N, L)
            N   = batch.shape[0]
            obs = batch.T.astype(np.float64)  # (L, N) — easier for time-first indexing

            # Emissions b[t, student, state]  shape (L, N, 2)
            b = np.empty((L, N, 2))
            b[:, :, 0] = np.where(obs == 1, guess,    1 - guess)
            b[:, :, 1] = np.where(obs == 1, 1 - slip, slip)

            # ── Forward pass (scaled Rabiner) ──────────────────────────────
            alpha = np.empty((L, N, 2))
            sc    = np.empty((L, N))

            alpha[0, :, 0] = (1 - prior) * b[0, :, 0]
            alpha[0, :, 1] =      prior  * b[0, :, 1]
            sc[0] = alpha[0].sum(axis=1)
            ok = sc[0] > 0
            alpha[0, ok] /= sc[0, ok, None]

            for t in range(1, L):
                alpha[t, :, 0] = b[t, :, 0] * alpha[t-1, :, 0] * A00
                alpha[t, :, 1] = b[t, :, 1] * (alpha[t-1, :, 0] * A01
                                                + alpha[t-1, :, 1])
                sc[t] = alpha[t].sum(axis=1)
                ok = sc[t] > 0
                alpha[t, ok] /= sc[t, ok, None]

            total_ll += np.sum(np.log(sc + 1e-300))

            # ── Backward pass (same scaling) ───────────────────────────────
            beta = np.empty((L, N, 2))
            ok_last = sc[L-1] > 0
            beta[L-1] = 1.0
            beta[L-1, ok_last] /= sc[L-1, ok_last, None]

            for t in range(L - 2, -1, -1):
                beta[t, :, 0] = (A00 * b[t+1, :, 0] * beta[t+1, :, 0]
                                 + A01 * b[t+1, :, 1] * beta[t+1, :, 1])
                beta[t, :, 1] = b[t+1, :, 1] * beta[t+1, :, 1]
                ok = sc[t] > 0
                beta[t, ok] /= sc[t, ok, None]

            # ── State posteriors γ  shape (L, N, 2) ───────────────────────
            gamma = alpha * beta
            gs    = gamma.sum(axis=2, keepdims=True)
            gamma = np.where(gs > 0, gamma / np.where(gs > 0, gs, 1.0), gamma)

            # ── Transition posteriors ξ (no-forgetting: ξ[1→0] = 0) ──────
            if L > 1:
                xi00 = alpha[:-1,:,0] * A00 * b[1:,:,0] * beta[1:,:,0]
                xi01 = alpha[:-1,:,0] * A01 * b[1:,:,1] * beta[1:,:,1]
                xi11 = alpha[:-1,:,1]        * b[1:,:,1] * beta[1:,:,1]
                z    = xi00 + xi01 + xi11
                ok   = z > 0
                xi01n = np.where(ok, xi01 / np.where(ok, z, 1.0), 0.0)
                xi00n = np.where(ok, xi00 / np.where(ok, z, 1.0), 0.0)
                sum_xi_01        += xi01n.sum()
                sum_gamma0_trans += (xi00n + xi01n).sum()

            sum_prior_post += gamma[0, :, 1].sum()
            n_seq          += N
            sum_total_nl   += gamma[:, :, 0].sum()
            sum_total_l    += gamma[:, :, 1].sum()
            sum_correct_nl += gamma[:, :, 0][obs == 1].sum()
            sum_wrong_l    += gamma[:, :, 1][obs == 0].sum()

        # ── M-step ────────────────────────────────────────────────────────
        prior = float(np.clip(sum_prior_post  / max(n_seq, 1),              0.01,  0.99))
        learn = float(np.clip(sum_xi_01       / max(sum_gamma0_trans, 1e-12), 0.001, 0.99))
        guess = float(np.clip(sum_correct_nl  / max(sum_total_nl, 1e-12),   0.001, 0.4))
        slip  = float(np.clip(sum_wrong_l     / max(sum_total_l,  1e-12),   0.001, 0.4))

        if abs(total_ll - prev_ll) < tol:
            break
        prev_ll = total_ll

    return {'prior': prior, 'learn': learn, 'guess': guess, 'slip': slip,
            'n_sequences': n_sequences, 'n_obs': n_obs_total,
            'log_likelihood': total_ll, 'iterations': it + 1}


# ─── fit_bkt ──────────────────────────────────────────────────────────────────
def fit_bkt(df, n_iter=100, min_attempts_per_student=2, n_jobs=-1):
    """Fit BKT for all skills in parallel (n_jobs=-1 = all cores).

    Uses thread-based parallelism — NumPy releases the GIL during heavy
    array computation, so threads actually run concurrently here.
    """
    def _fit_one(skill):
        skill_df = df[df['skill_name'] == skill].sort_values(['user_id', 'order_id'])
        sequences = [
            group['correct'].tolist()
            for _, group in skill_df.groupby('user_id')
            if len(group) >= min_attempts_per_student
        ]
        if not sequences:
            return skill, None
        return skill, fit_skill(sequences, n_iter=n_iter)

    skills  = df['skill_name'].unique()
    results = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_fit_one)(sk) for sk in skills
    )
    return {sk: p for sk, p in results if p is not None}


# ─── predict_bkt ──────────────────────────────────────────────────────────────
def predict_bkt(df, params):
    """Generate per-attempt P(correct) using the fitted params."""
    df = df.copy().sort_values(['user_id', 'skill_name', 'order_id'])
    df['correct_predictions'] = np.nan
    for (user, skill), group in df.groupby(['user_id', 'skill_name']):
        if skill not in params:
            continue
        p = params[skill]
        preds = forward_pass(group['correct'].tolist(),
                             prior=p['prior'], learn=p['learn'],
                             guess=p['guess'], slip=p['slip'])
        df.loc[group.index, 'correct_predictions'] = preds
    return df

## 3. Fit & predict

In [ ]:
# NOTE: Identical EM maths to the analysis notebook - only the execution path changed
#       (length-batched vectorisation + joblib threads), so params/predictions match.
#       `elapsed` below is the wall-clock fit time for all skills.

import time

print("Fitting BKT for", df_bkt['skill_name'].nunique(), "skills...")
t0 = time.perf_counter()
bkt_params = fit_bkt(df_bkt, n_iter=100, min_attempts_per_student=2, n_jobs=-1)
elapsed = time.perf_counter() - t0
print(f"Done — {len(bkt_params)} skills fitted in {elapsed:.1f}s")

params_df = pd.DataFrame(bkt_params).T.round(4)
params_df.index.name = 'skill'
params_df.head(10)

In [ ]:
df_predictions = predict_bkt(df_bkt, bkt_params)
df_predictions[['user_id', 'skill_name', 'correct', 'correct_predictions']].head(10)

## 4. Evaluation metrics

In [ ]:
# NOTE: Same metrics as the analysis notebook (AUC / RMSE + identifiability check).

valid = df_predictions.dropna(subset=['correct_predictions'])

auc  = roc_auc_score(valid['correct'], valid['correct_predictions']) if valid['correct'].nunique() == 2 else float('nan')
rmse = mean_squared_error(valid['correct'], valid['correct_predictions']) ** 0.5

print(f"Overall AUC:   {auc:.4f}")
print(f"Overall RMSE:  {rmse:.4f}")
print(f"N predictions: {len(valid):,}")

print("\nParameter distribution across skills:")
print(params_df[['prior', 'learn', 'guess', 'slip']].describe().round(3))

print("\nIdentifiability check (guess + slip should be < 1):")
print((params_df['guess'] + params_df['slip']).describe().round(3))

In [ ]:
skill_metrics = (
    valid.groupby('skill_name')
    .apply(lambda g: pd.Series({
        'auc':  roc_auc_score(g['correct'], g['correct_predictions'])
                if g['correct'].nunique() == 2 else np.nan,
        'rmse': mean_squared_error(g['correct'], g['correct_predictions']) ** 0.5,
        'n':    len(g),
    }))
    .reset_index()
)
skill_metrics.sort_values('auc', ascending=False).head(10)

## 5. Visualisations

In [ ]:
# ── 1. Parameter distributions ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col, color in zip(
    axes.flat,
    ['prior', 'learn', 'guess', 'slip'],
    ['#4C72B0', '#55A868', '#C44E52', '#8172B2'],
):
    ax.hist(params_df[col], bins=30, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(params_df[col].mean(), color='black', linestyle='--', linewidth=1,
               label=f"mean={params_df[col].mean():.3f}")
    ax.set_title(f"{col}  (n={len(params_df)} skills)")
    ax.set_xlabel(col); ax.set_ylabel('number of skills')
    ax.legend(loc='upper right', frameon=False)
fig.suptitle("BKT parameter distributions across skills", y=1.02, fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
# ── 2. Guess vs. slip (identifiability) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sizes = np.log1p(params_df['n_obs']) * 8 if 'n_obs' in params_df.columns else 40
ax.scatter(params_df['guess'], params_df['slip'],
           s=sizes, alpha=0.55, c='#4C72B0', edgecolor='white', linewidth=0.5)
xs = np.linspace(0, 0.5, 50)
ax.plot(xs, 1 - xs, '--', color='red',  alpha=0.5, label='guess + slip = 1')
ax.plot([0, 0.5], [0, 0.5], ':',  color='gray', alpha=0.5, label='guess = slip')
ax.set_xlabel('guess'); ax.set_ylabel('slip')
ax.set_xlim(0, 0.5);   ax.set_ylim(0, 0.5)
ax.set_title("Guess vs. slip per skill")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# ── 3. Empirical learning curves (top 6 skills) ────────────────────────────
top_skills = df_bkt['skill_name'].value_counts().head(6).index.tolist()
fig, ax = plt.subplots(figsize=(10, 6))
for skill in top_skills:
    sub = df_bkt[df_bkt['skill_name'] == skill].copy()
    sub['skill_attempt'] = sub.groupby('user_id').cumcount() + 1
    curve = sub.groupby('skill_attempt')['correct'].agg(['mean', 'count'])
    curve = curve[curve['count'] >= 20]
    ax.plot(curve.index[:30], curve['mean'].values[:30],
            marker='o', markersize=4, label=f"skill {skill} (n={len(sub):,})")
ax.set_xlabel('attempt number on this skill')
ax.set_ylabel('P(correct) — empirical mean')
ax.set_title("Empirical learning curves — 6 most-practised skills")
ax.legend(frameon=False, loc='lower right'); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Mastery trajectory for one student–skill pair ───────────────────────
candidates = (df_bkt.groupby(['user_id', 'skill_name']).size()
              .reset_index(name='n').query('15 <= n <= 40')
              .sort_values('n', ascending=False))

if len(candidates) > 0:
    pick = candidates.iloc[11]
    u, sk = int(pick['user_id']), pick['skill_name']
    sub = (df_bkt[(df_bkt['user_id'] == u) & (df_bkt['skill_name'] == sk)]
           .sort_values('order_id'))
    obs = sub['correct'].tolist()
    p   = bkt_params[sk]

    p_l_trace, p_l = [], p['prior']
    for c in obs:
        num = p_l * (1 - p['slip']) if c == 1 else p_l * p['slip']
        den = (num + (1 - p_l) * p['guess']) if c == 1 else (num + (1 - p_l) * (1 - p['guess']))
        p_l = (num / den if den > 0 else p_l)
        p_l = p_l + (1 - p_l) * p['learn']
        p_l_trace.append(p_l)

    fig, ax = plt.subplots(figsize=(11, 4.5))
    xs = np.arange(1, len(obs) + 1); obs_arr = np.array(obs)
    ax.plot(xs, p_l_trace, '-', color='#4C72B0', linewidth=2, label='P(learned)')
    ax.scatter(xs[obs_arr == 1], np.array(p_l_trace)[obs_arr == 1],
               marker='o', s=70, color='#55A868', label='correct', zorder=5)
    ax.scatter(xs[obs_arr == 0], np.array(p_l_trace)[obs_arr == 0],
               marker='x', s=70, color='#C44E52', label='wrong',   zorder=5)
    ax.axhline(0.95, linestyle='--', color='gray', alpha=0.5, label='mastery (0.95)')
    ax.set_xlabel('attempt #'); ax.set_ylabel('P(learned)')
    ax.set_title(f"Mastery trajectory  user={u}, skill={sk}\n"
                 f"prior={p['prior']:.2f}  learn={p['learn']:.2f}  "
                 f"guess={p['guess']:.2f}  slip={p['slip']:.2f}")
    ax.set_ylim(0, 1.05); ax.legend(frameon=False, loc='lower right')
    plt.tight_layout(); plt.show()
else:
    print("No candidate pair with 15–40 attempts found.")

In [ ]:
# ── 5. Calibration plot ─────────────────────────────────────────────────────
bins = np.linspace(0, 1, 11)
valid2 = valid.copy()
valid2['pred_bin'] = pd.cut(valid2['correct_predictions'], bins, include_lowest=True)
cal = valid2.groupby('pred_bin', observed=True).agg(
    pred_mean=('correct_predictions', 'mean'),
    obs_mean =('correct',             'mean'),
    n        =('correct',             'size'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='perfect calibration')
ax.scatter(cal['pred_mean'], cal['obs_mean'],
           s=np.sqrt(cal['n']) * 1.5, alpha=0.75,
           color='#4C72B0', edgecolor='white', linewidth=0.5)
ax.plot(cal['pred_mean'], cal['obs_mean'], '-', color='#4C72B0', alpha=0.6)
for _, row in cal.iterrows():
    ax.annotate(f"n={int(row['n']):,}", (row['pred_mean'], row['obs_mean']),
                xytext=(5, -8), textcoords='offset points', fontsize=8, color='gray')
ax.set_xlabel('mean predicted P(correct)')
ax.set_ylabel('observed fraction correct')
ax.set_title("Calibration plot (decile bins)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout(); plt.show()